In [6]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import polars as pl
import math
from typing import Dict, List, Optional

sys.path.append('../')


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
df_train = pl.read_parquet('../../data/clustered/train_cluster_features.parquet')
df_val = pl.read_parquet('../../data/clustered/val_cluster_features.parquet')
df_test = pl.read_parquet('../../data/clustered/test_cluster_features.parquet')


print(f"Train shape: {df_train.shape}")
print(f"Val shape: {df_val.shape}")
print(f"Test shape: {df_test.shape}")


Train shape: (4892637, 225)
Val shape: (808077, 225)
Test shape: (1906221, 225)


In [ ]:
df_train.head()

cluster_id,ts_start,dep_internal_count,dep_internal_male,dep_internal_female,dep_internal_other,dep_internal_young,dep_internal_adult,dep_internal_senior,dep_internal_age_available,dep_internal_model_iconic,dep_internal_duration_mean,dep_internal_duration_std,dep_internal_temperature_2m_mean,dep_internal_relative_humidity_2m_mean,dep_internal_dew_point_2m_mean,dep_internal_apparent_temperature_mean,dep_internal_precipitation_mean,dep_internal_weather_code_mean,dep_internal_pressure_msl_mean,dep_internal_cloud_cover_mean,dep_internal_cloud_cover_low_mean,dep_internal_cloud_cover_mid_mean,dep_internal_cloud_cover_high_mean,dep_internal_vapour_pressure_deficit_mean,dep_internal_wind_speed_10m_mean,dep_internal_wind_direction_10m_mean,dep_internal_wind_gusts_10m_mean,dep_internal_soil_temperature_0_to_7cm_mean,dep_internal_soil_moisture_0_to_7cm_mean,dep_internal_sunshine_duration_mean,dep_internal_direct_radiation_mean,dep_internal_temp_feel_diff_mean,dep_internal_weather_comfort_mean,dep_internal_is_day_count,dep_internal_temp_comfortable_count,dep_internal_temp_very_hot_count,…,is_morning_rush,is_evening_rush,is_daytime,is_summer,is_winter,temperature_2m_mean,relative_humidity_2m_mean,dew_point_2m_mean,apparent_temperature_mean,precipitation_mean,weather_code_mean,pressure_msl_mean,cloud_cover_mean,cloud_cover_low_mean,cloud_cover_mid_mean,cloud_cover_high_mean,vapour_pressure_deficit_mean,wind_speed_10m_mean,wind_direction_10m_mean,wind_gusts_10m_mean,soil_temperature_0_to_7cm_mean,soil_moisture_0_to_7cm_mean,sunshine_duration_mean,direct_radiation_mean,temp_feel_diff_mean,weather_comfort_mean,is_day,temp_comfortable,temp_very_hot,temp_cold,is_raining,heavy_rain,light_rain,strong_wind,moderate_wind,weather_matched,weather_data_available
i32,datetime[μs],u32,u32,u32,u32,i32,i32,i32,i32,i32,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,i32,i32,i32,…,i32,i32,i32,i32,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32
0,2020-01-01 00:00:00,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,…,0,0,0,1,0,21.311012,87.557724,19.161009,22.377903,0.0,1.0,1010.700745,39.0,12.0,19.0,23.0,0.315351,16.363167,140.355743,27.719978,23.110994,0.468,0.0,0.0,1.066872,1.0,0,1,0,0,0,0,0,0,1,1,1
0,2020-01-01 00:30:00,1,1,0,0,0,0,0,0,1,3599.0,0.0,21.311001,87.557816,19.161001,22.377872,0.0,1.0,1010.700012,39.0,12.0,19.0,23.0,0.315351,16.363178,140.355865,27.719999,23.111,0.468,0.0,0.0,1.066872,1.0,0,1,0,…,0,0,0,1,0,21.311024,87.557831,19.160957,22.377916,0.0,1.0,1010.702087,39.0,12.0,19.0,23.0,0.315351,16.363216,140.355591,27.720058,23.110992,0.468001,0.0,0.0,1.066868,1.0,0,1,0,0,0,0,0,0,1,1,1
0,2020-01-01 01:00:00,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,…,0,0,0,1,0,21.311026,87.55793,19.16098,22.377918,0.0,1.0,1010.702209,39.0,12.0,19.0,23.0,0.315351,16.363224,140.355576,27.720079,23.110973,0.468001,0.0,0.0,1.066868,1.0,0,1,0,0,0,0,0,0,1,1,1
0,2020-01-01 01:30:00,2,2,0,0,0,0,0,0,2,1710.5,26.162951,21.311001,87.557816,19.161001,22.377872,0.0,1.0,1010.700012,39.0,12.0,19.0,23.0,0.315351,16.363178,140.355865,27.719999,23.111,0.468,0.0,0.0,1.066872,1.0,0,2,0,…,0,0,0,1,0,21.311026,87.557922,19.160978,22.377918,0.0,1.0,1010.702209,39.0,12.0,19.0,23.0,0.315351,16.363224,140.355576,27.720078,23.110975,0.468001,0.0,0.0,1.066868,1.0,0,1,0,0,0,0,0,0,1,1,1
0,2020-01-01 02:00:00,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,…,0,0,0,1,0,21.311026,87.557899,19.160973,22.377918,0.0,1.0,1010.702209,39.0,12.0,19.0,23.0,0.315351,16.363222,140.355576,27.720074,23.110979,0.468001,0.0,0.0,1.066868,1.0,0,1,0,0,0,0,0,0,1,1,1


In [9]:
df_train.columns

['cluster_id',
 'ts_start',
 'dep_internal_count',
 'dep_internal_male',
 'dep_internal_female',
 'dep_internal_other',
 'dep_internal_young',
 'dep_internal_adult',
 'dep_internal_senior',
 'dep_internal_age_available',
 'dep_internal_model_iconic',
 'dep_internal_duration_mean',
 'dep_internal_duration_std',
 'dep_internal_temperature_2m_mean',
 'dep_internal_relative_humidity_2m_mean',
 'dep_internal_dew_point_2m_mean',
 'dep_internal_apparent_temperature_mean',
 'dep_internal_precipitation_mean',
 'dep_internal_weather_code_mean',
 'dep_internal_pressure_msl_mean',
 'dep_internal_cloud_cover_mean',
 'dep_internal_cloud_cover_low_mean',
 'dep_internal_cloud_cover_mid_mean',
 'dep_internal_cloud_cover_high_mean',
 'dep_internal_vapour_pressure_deficit_mean',
 'dep_internal_wind_speed_10m_mean',
 'dep_internal_wind_direction_10m_mean',
 'dep_internal_wind_gusts_10m_mean',
 'dep_internal_soil_temperature_0_to_7cm_mean',
 'dep_internal_soil_moisture_0_to_7cm_mean',
 'dep_internal_sunshi